In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import os

import sklearn.utils as su
import sklearn.metrics as sm
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Lasso
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

import xgboost as xgb
import lightgbm as lgb

In [ ]:
# ✅ 设置字体
matplotlib.rcParams['font.family'] = 'Liberation Serif'
matplotlib.rcParams['axes.unicode_minus'] = False    # 正常显示负号

In [ ]:
# ------------------- 设置CPU线程数，解决LGBM警告 -------------------
os.environ["LOKY_MAX_CPU_COUNT"] = "8"

In [ ]:
# 第一步：导入数据
file_path = []
df = pd.read_excel(file_path)

In [ ]:
# ------------------- 选择特征和目标列 -------------------
features = []
target_column = []

x = df[features]
y = df[target_column]

In [ ]:
# ------------------- 归一化处理 -------------------
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()
x_scaled = scaler_x.fit_transform(x)
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))
x_scaled = pd.DataFrame(x_scaled, columns=x.columns)

In [ ]:
# ------------------- 数据打乱拆分 -------------------
x, y = su.shuffle(x_scaled, y_scaled, random_state=25)
train_size = int(len(x) * 0.8)
train_x, test_x = x[:train_size], x[train_size:]
train_y, test_y = y[:train_size], y[train_size:]

In [ ]:
# ------------------- 存储所有模型最优参数 -------------------
best_param_dict = {}

In [ ]:
#模型评估函数 
def evaluate_model(model, train_x, train_y, test_x, test_y, model_name):
    model.fit(train_x, train_y.ravel())
    train_pred = model.predict(train_x)
    test_pred = model.predict(test_x)

    train_pred_inv = scaler_y.inverse_transform(train_pred.reshape(-1, 1))
    test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1))
    train_y_inv = scaler_y.inverse_transform(train_y.reshape(-1, 1))
    test_y_inv = scaler_y.inverse_transform(test_y.reshape(-1, 1))

    def calculate_metrics(y_true, y_pred):
        mse = sm.mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mae = sm.mean_absolute_error(y_true, y_pred)
        mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        r2 = sm.r2_score(y_true, y_pred)
        return mse, rmse, mae, mape, r2

    train_mse, train_rmse, train_mae, train_mape, train_r2 = calculate_metrics(train_y_inv, train_pred_inv)
    test_mse, test_rmse, test_mae, test_mape, test_r2 = calculate_metrics(test_y_inv, test_pred_inv)

    print(f"\n=== {model_name} Evaluation ===")
    print(f"Train Set: MSE={train_mse:.4f}, RMSE={train_rmse:.4f}, MAE={train_mae:.4f}, MAPE={train_mape:.2f}%, R²={train_r2:.4f}")
    print(f"Test Set:  MSE={test_mse:.4f}, RMSE={test_rmse:.4f}, MAE={test_mae:.4f}, MAPE={test_mape:.2f}%, R²={test_r2:.4f}")

    plt.figure(figsize=(14, 6))

    # Training Set Plot
    plt.subplot(1, 2, 1)
    plt.plot(train_y_inv, label='True Value', linewidth=2)
    plt.plot(train_pred_inv, label='Predicted Value', linestyle='--')
    plt.title(f"{model_name} - Training Set")
    plt.xlabel("Sample Index")
    plt.ylabel("Main Engine Fuel Consumption")
    plt.legend(loc='upper right')  # 图例放右上角
    plt.grid(True)

    # Test Set Plot
    plt.subplot(1, 2, 2)
    plt.plot(test_y_inv, label='True Value', linewidth=2)
    plt.plot(test_pred_inv, label='Predicted Value', linestyle='--')
    plt.title(f"{model_name} - Test Set")
    plt.xlabel("Sample Index")
    plt.ylabel("Main Engine Fuel Consumption")
    plt.legend(loc='upper right')  # 图例放右上角
    plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
# ------------------- 网格搜索函数 -------------------
def grid_search_and_evaluate(model, param_grid, model_name):
    print(f"\n📌 正在进行 {model_name} 的网格搜索...")
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )
    grid_search.fit(train_x, train_y.ravel())

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    print(f"\n✅ {model_name} 最佳参数如下：")
    for key, val in best_params.items():
        print(f"  - {key}: {val}")

    best_param_dict[model_name] = best_params
    evaluate_model(best_model, train_x, train_y, test_x, test_y, model_name)

In [ ]:
# 随机森林
rf_param_grid = {
    'n_estimators': [100, 150, 200, 250, 300],
    'max_depth': [3, 5, 7, 9, 11, 13, 15],
    'min_samples_split': [2, 5, 7, 9],
    'min_samples_leaf': [1, 2, 3, 4, 5]
}
grid_search_and_evaluate(RandomForestRegressor(random_state=20), rf_param_grid, "Random Forest")

In [ ]:
# 决策树
dt_param_grid = {
    'max_depth': [3, 5, 7, 9, 11, 13, 15],
    'min_samples_split': [2, 5, 7, 9],
    'min_samples_leaf': [1, 2, 3, 4, 5]
}
grid_search_and_evaluate(DecisionTreeRegressor(random_state=25), dt_param_grid, "Decision Tree")

In [ ]:
# XGBoost
xgb_param_grid = {
    'n_estimators': [100, 150, 200, 250, 300],
    'max_depth': [3, 5, 7, 9, 11, 13, 15],
    'learning_rate': [0.05, 0.1, 0.15, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}
grid_search_and_evaluate(
    xgb.XGBRegressor(objective="reg:squarederror", random_state=25, enable_categorical=False),
    xgb_param_grid, "XGBoost"
)

In [ ]:
# LightGBM
lgb_param_grid = {
    'n_estimators': [100, 150, 200, 250, 300],
    'learning_rate': [0.05, 0.1, 0.15, 0.2],
    'max_depth': [3, 5, 7, 9, 11, 13, 15],
    'num_leaves': [10, 20, 30, 40],
    'min_data_in_leaf': [10, 20, 30],
    'feature_fraction': [0.8, 1.0],
    'bagging_fraction': [0.8, 1.0],
    'bagging_freq': [1],
    'lambda_l1': [0.0],
    'lambda_l2': [0.01],
    'verbosity': [-1]
}
grid_search_and_evaluate(lgb.LGBMRegressor(objective='regression', random_state=25), lgb_param_grid, "LightGBM")

In [ ]:
# AdaBoost
ab_param_grid = {
    'n_estimators': [100, 150, 200, 250, 300],
    'learning_rate': [0.05, 0.1, 0.15, 0.2],
    'estimator__max_depth': [3, 5, 7, 9, 11, 13, 15],
    'estimator__min_samples_split': [2, 5, 7, 9],
    'estimator__min_samples_leaf': [1, 2, 3, 4, 5],
    'loss':['linear', 'square', 'exponential']
}
base_tree = DecisionTreeRegressor(random_state=25)
grid_search_and_evaluate(AdaBoostRegressor(estimator=base_tree, random_state=25), ab_param_grid, "AdaBoost")

In [ ]:
# LASSO
lasso_param_grid = {
    'alpha': [0.0001, 0.001, 0.01]
}
grid_search_and_evaluate(Lasso(), lasso_param_grid, "LASSO")

In [ ]:
# 支持向量回归 SVR
svr_param_grid = {
    'C': [1, 10],
    'epsilon': [0.01, 0.1],
    'kernel': ['rbf']
}
grid_search_and_evaluate(SVR(), svr_param_grid, "SVR")

In [ ]:
# ------------------- 所有最优参数输出 -------------------
print("\n📦 所有模型的最优参数汇总：")
for model_name, params in best_param_dict.items():
    print(f"\n🔹 {model_name}:")
    for key, val in params.items():
        print(f"  - {key}: {val}")

<font size="10">ANN

In [ ]:
# ✅ 自定义 PyTorch 模型包装器（兼容 GridSearch）
class SimpleANN(BaseEstimator, RegressorMixin):
    def __init__(self, input_dim=None, hidden_dim=32, activation='relu', lr=0.01, batch_size=32, epochs=100, device='cpu'):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.activation = activation
        self.lr = lr
        self.batch_size = batch_size
        self.epochs = epochs
        self.device = device
        self.model = None

    def get_activation(self):
        if self.activation == 'relu':
            return nn.ReLU()
        elif self.activation == 'tanh':
            return nn.Tanh()
        elif self.activation == 'sigmoid':
            return nn.Sigmoid()
        else:
            raise ValueError(f"Unsupported activation: {self.activation}")

    def build_model(self):
        act = self.get_activation()
        model = nn.Sequential(
            nn.Linear(self.input_dim, self.hidden_dim),
            act,
            nn.Linear(self.hidden_dim, 1)
        )
        return model.to(self.device)

    def fit(self, X, y):
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)
        dataset = TensorDataset(X, y)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        self.model = self.build_model()
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.MSELoss()

        self.model.train()
        for epoch in range(self.epochs):
            for batch_x, batch_y in loader:
                batch_x, batch_y = batch_x.to(self.device), batch_y.to(self.device)
                optimizer.zero_grad()
                output = self.model(batch_x).squeeze()
                loss = criterion(output, batch_y)
                loss.backward()
                optimizer.step()
        return self

    def predict(self, X):
        self.model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            preds = self.model(X_tensor).cpu().numpy()
        return preds

In [ ]:
# ✅ 超参数搜索空间
param_grid = {
    'input_dim': [train_x.shape[1]],
    'hidden_dim': [8, 16, 32, 64],           # ✅ 隐藏层神经元数
    'activation': ['relu', 'tanh', 'sigmoid'],  # ✅ 激活函数
    'lr': [0.01, 0.02, 0.03],
    'batch_size': [16, 32, 64],
    'epochs': [100, 150, 200, 250, 300]
}

In [ ]:
# ✅ 网格搜索 + 5折交叉验证
model = SimpleANN(device='cuda' if torch.cuda.is_available() else 'cpu')
grid = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_error', verbose=1, n_jobs=1)
grid.fit(train_x.values, train_y.ravel())
print("\n✅ 最优超参数组合：")
print(grid.best_params_)